# **Fingerprint in CrewAI**

## Overview

Fingerprints in CrewAI provide a way to **uniquely identify and track components throughout their lifecycle**.

Each **Agent**, **Crew**, and **Task** automatically receives a unique fingerprint when created, which cannot be manually overridden.


These fingerprints can be used for:

 > Auditing and tracking component usage

 > Ensuring component identity integrity

 > Attaching metadata to components

 > Creating a traceable chain of operations


## How Fingerprints Work

A fingerprint is an instance of the Fingerprint class from the crewai.security module. Each fingerprint contains:

A **UUID string**: A unique identifier for the component that is automatically generated and cannot be manually set

A creation **timestamp**: When the fingerprint was generated, automatically set and cannot be manually modified

**Metadata**: A dictionary of additional information that can be customized

**Fingerprints are automatically generated** and assigned when a component is created.

Each component exposes its fingerprint through a **read-only property**.

## Install necessary Libraries

In [ ]:
!pip install -q crewai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.8/195.8 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9

In [ ]:
import crewai
print(crewai.__version__)

1.15.17


# Set API Keys

In [ ]:
from google.colab import userdata
import os
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

## Import Dependencies

In [ ]:
from crewai import Agent, Task, Crew, LLM

## Create the LLM Object

In [ ]:
# OPENROUTER hosted LLMs
llm = LLM(
    model="openrouter/openai/gpt-oss-120b",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)


# Define Agents

In [ ]:
from crewai import Agent, Crew, Task

# Create components - fingerprints are automatically generated
agent = Agent(
    role="Data Scientist",
    goal="Analyze data",
    backstory="Expert in data analysis",
    llm=llm
)

crew = Crew(
    agents=[agent],
    tasks=[]
)

task = Task(
    description="Analyze customer data",
    expected_output="Insights from data analysis",
    agent=agent
)


## Access Fingerprints

In [ ]:
# Access the fingerprints
agent_fingerprint = agent.fingerprint
crew_fingerprint = crew.fingerprint
task_fingerprint = task.fingerprint

# Print the UUID strings
print(f"Agent fingerprint: {agent_fingerprint.uuid_str}")
print(f"Crew fingerprint: {crew_fingerprint.uuid_str}")
print(f"Task fingerprint: {task_fingerprint.uuid_str}")

Agent fingerprint: e7bc9035-28e6-4cb0-9e9f-a1ad9aae3b60
Crew fingerprint: d263f67e-5802-43c2-85c4-e55c8ab4c5b8
Task fingerprint: c00fa0a7-c8d6-462d-9541-25f9488b885c


In [ ]:
# agent_fingerprint.to_dict()

## Working with Fingerprint Metadata

You can add metadata to fingerprints for additional context:

In [ ]:
# Add metadata to the agent's fingerprint
agent.security_config.fingerprint.metadata = {
    "version": "1.0",
    "department": "Data Science",
    "project": "Customer Analysis"
}

# Access the metadata
print(f"Agent metadata: {agent.fingerprint.metadata}")

Agent metadata: {'version': '1.0', 'department': 'Data Science', 'project': 'Customer Analysis'}



## Fingerprint Persistence

Fingerprints are designed to persist and remain unchanged throughout a component’s lifecycle. If you modify a component, the fingerprint remains the same:

In [ ]:
original_fingerprint = agent.fingerprint.uuid_str

# Modify the agent
agent.goal = "New goal for analysis"

In [ ]:
# The fingerprint remains unchanged
assert agent.fingerprint.uuid_str == original_fingerprint

## Deterministic Fingerprints

While you cannot directly set the UUID and creation timestamp, you can create deterministic fingerprints using the generate method with a seed:

In [ ]:
from crewai.security import Fingerprint

# Create a deterministic fingerprint using a seed string
deterministic_fingerprint = Fingerprint.generate(seed="my-agent-id")

# The same seed always produces the same fingerprint
same_fingerprint = Fingerprint.generate(seed="my-agent-id")
assert deterministic_fingerprint.uuid_str == same_fingerprint.uuid_str

# You can also set metadata
custom_fingerprint = Fingerprint.generate(
    seed="my-agent-id",
    metadata={"version": "1.0"}
)

## Advanced Usage - Fingerprint Structure

Each fingerprint has the following structure:

In [ ]:
from crewai.security import Fingerprint

fingerprint = agent.fingerprint

# UUID string - the unique identifier (auto-generated)
uuid_str = fingerprint.uuid_str  # e.g., "123e4567-e89b-12d3-a456-426614174000"

# Creation timestamp (auto-generated)
created_at = fingerprint.created_at  # A datetime object

# Metadata - for additional information (can be customized)
metadata = fingerprint.metadata  # A dictionary, defaults to {}

In [ ]:
print(uuid_str)

print(created_at)

print(metadata)

e7bc9035-28e6-4cb0-9e9f-a1ad9aae3b60
2026-08-24 16:48:44.583161
{'version': '1.0', 'department': 'Data Science', 'project': 'Customer Analysis'}
